In [1]:
!pip install gensim
!pip install nltk

In [5]:
from google.colab import drive
!git clone https://github.com/fabianagoes/ismb_tutorial8.git
%cd ismb_tutorial8
drive.mount('/content/drive')
import os
from gensim.models import Word2Vec
import pandas as pd
from utils import *
import ipywidgets as widgets
from IPython.display import display

Cloning into 'ismb_tutorial8'...
remote: Enumerating objects: 263, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 263 (delta 42), reused 43 (delta 19), pack-reused 193 (from 1)
Receiving objects: 100% (263/263), 54.05 MiB | 14.85 MiB/s, done.
Resolving deltas: 100% (78/78), done.
Updating files: 100% (106/106), done.
/content/ismb_tutorial8/ismb_tutorial8
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
root_dir = "/content/ismb_tutorial8/datasets"
subdirs = get_all_subdirs(root_dir) # internal function that goes over all subdirectories. Source code: utils.py

dropdown = widgets.Dropdown(
    options=subdirs,
    description='Select:',
    disabled=False,
)

selected_path = {'value': subdirs[0]}

def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected_path['value'] = change['new']

dropdown.observe(on_change)

display(dropdown)

Dropdown(description='Select:', options=('/content/ismb_tutorial8/datasets/GUE/virus/covid', '/content/ismb_tu…

In [14]:
print('Dataset Selected: ',selected_path['value'])

train_data=pd.read_csv(selected_path['value']+'/train.csv')#read_csv to read csv files
val_data=pd.read_csv(selected_path['value']+'/dev.csv')
test_data=pd.read_csv(selected_path['value']+'/test.csv')
data={'Train':train_data,'Val':val_data,'Test':test_data}
print("\nDetailed information:")
for name, dataset in data.items():
    print(f"{name}: {dataset.shape[0]} rows, {dataset.shape[1]} col")

print("\nShowing the first 5 train datapoints:")
train_data.head(5)

Dataset Selected:  /content/ismb_tutorial8/datasets/GUE/prom/prom_300_tata

Detailed information:
Train: 4904 rows, 2 col
Val: 613 rows, 2 col
Test: 613 rows, 2 col

Showing the first 5 train datapoints:


,sequence,label
0,CCAGGTCCCGGGAGCGCCACGGAACCTAACGGTGGCAGCGGAGGTC...,1
1,CCGGCCGAGCTCAGCACCGAGGCGCCCCCCAACCTGCCCAGCCCCC...,0
2,CACCTTGGACTTGGGACCAGAAAGAGGTGGGTTGGGTGAAGAGGCA...,1
3,CCTCCCTCCCCTAGAACACCCTCCTCATACCTGGCCTTTGGTCTTC...,1
4,CAATTTTGATTATCTGACAACATGATGTCCAAACTGAGGTATTTAT...,1


In [18]:
# Function to generate k-mers
def kmerize(sequence, k=3):
    return [sequence[i:i+k] for i in range(len(sequence)-k+1)]

# Create corpus for word2vec using training sequences
k = 3
corpus = [kmerize(seq, k) for seq in train_data['sequence']]

print(f"First {3} k-merized sequences (k={k}):")
print("-" * 60)

for i, kmers in enumerate(corpus[:3]):
    print(f"Sequence {i+1}:")
    print(f"  K-mers: {kmers}")
    print(f"  Number of k-mers: {len(kmers)}")
    print(f"  First 5 k-mers: {kmers[:5]}")
    print("-" * 60)

First 3 k-merized sequences (k=3):
------------------------------------------------------------
Sequence 1:
  K-mers: ['CCA', 'CAG', 'AGG', 'GGT', 'GTC', 'TCC', 'CCC', 'CCG', 'CGG', 'GGG', 'GGA', 'GAG', 'AGC', 'GCG', 'CGC', 'GCC', 'CCA', 'CAC', 'ACG', 'CGG', 'GGA', 'GAA', 'AAC', 'ACC', 'CCT', 'CTA', 'TAA', 'AAC', 'ACG', 'CGG', 'GGT', 'GTG', 'TGG', 'GGC', 'GCA', 'CAG', 'AGC', 'GCG', 'CGG', 'GGA', 'GAG', 'AGG', 'GGT', 'GTC', 'TCG', 'CGC', 'GCG', 'CGC', 'GCC', 'CCC', 'CCC', 'CCT', 'CTC', 'TCA', 'CAG', 'AGT', 'GTG', 'TGC', 'GCC', 'CCC', 'CCG', 'CGC', 'GCG', 'CGC', 'GCT', 'CTC', 'TCT', 'CTC', 'TCC', 'CCC', 'CCC', 'CCG', 'CGT', 'GTC', 'TCG', 'CGG', 'GGG', 'GGA', 'GAG', 'AGC', 'GCT', 'CTT', 'TTC', 'TCC', 'CCT', 'CTG', 'TGG', 'GGT', 'GTC', 'TCG', 'CGC', 'GCC', 'CCC', 'CCC', 'CCT', 'CTG', 'TGC', 'GCG', 'CGG', 'GGC', 'GCG', 'CGG', 'GGC', 'GCG', 'CGG', 'GGC', 'GCT', 'CTC', 'TCG', 'CGG', 'GGG', 'GGG', 'GGT', 'GTG', 'TGT', 'GTC', 'TCT', 'CTG', 'TGG', 'GGC', 'GCC', 'CCG', 'CGG', 'GGC', 'GCG', 'CGC',

In [19]:
# Train the word2vec model
w2v_model = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=1, workers=2, epochs=10)

print("Word2Vec Model Information:")
print("=" * 50)

# Informações básicas do modelo
print(f"Vocabulary size: {len(w2v_model.wv)}")
print(f"Vector dimensions: {w2v_model.wv.vector_size}")
print(f"Window size: {w2v_model.window}")
print(f"Min count: {w2v_model.min_count}")
print(f"Training epochs: {w2v_model.epochs}")

# Show vector of first k-mer of first sequence
example_kmer = corpus[0][0]
print(f"Vetor embedding para {example_kmer}:")
print(w2v_model.wv[example_kmer])

print("Number of learned k-mers:", len(w2v_model.wv))
print("Example of vectors for k-mer 'ATC':")
print(w2v_model.wv['ATC'])  # replace with a k-mer present in your dataset

# Show the most similar k-mers
print("K-mers mais similares a", example_kmer)
print(w2v_model.wv.most_similar(example_kmer))

Word2Vec Model Information:
Vocabulary size: 64
Vector dimensions: 100
Window size: 5
Min count: 1
Training epochs: 10
Vetor embedding para CCA:
[-0.69157946  1.6478288   0.37581474  0.18550344  1.4959493  -1.0525328
  1.5258166  -0.40922844 -0.78064966 -1.3109176  -1.1691744  -0.7352332
 -0.47753924  1.1493676   0.9307118  -0.41340068 -1.6213293   1.069984
 -1.2336135   0.2855425   0.40856764 -0.07287672  0.67550474  0.5709027
 -0.42930746 -0.8282983  -0.60467625 -0.42169997  0.8213426   0.7256356
  0.28635183 -0.91638213 -1.1085526  -0.4061358   1.0736965   1.3668253
 -0.8346981   1.7935458   0.6907943  -0.9024605   0.89112353 -0.343147
  0.05105967  0.76160187 -0.06159205 -0.9750214  -0.5949389  -1.5186723
  1.349588   -0.01827377 -0.6185184  -0.3534113   2.3944101  -1.2251992
  1.7739968   0.698675    1.18135     1.598751   -0.07244571  1.0844421
 -0.6396215  -0.43706825 -0.18935065 -0.3410173  -0.4460912  -0.452662
  0.36855125  1.3318919   1.0039144   0.14481537 -0.37711346 -1.03